# Generate charts for new songs

Charts any audio file (not just the test set) with one or more saved runs, and writes song folders a TJA simulator can load, in the same layout as `07_exports`:

```text
outputs/exports/<datetime>-<alias>/
├── export.json
└── My Song/
    ├── My Song.ogg
    ├── My Song (donderformer_base_oni_v2).tja     # TITLE: My Song (donderformer_base_oni_v2)
    └── My Song (taikonation_bs1_oni_v2).tja
```

Each chart's file name and in-game title carry its label, `<model>_<alias>_<split>` from the run's `config.json`. Works with notebook 03 (CNN-LSTM) and notebook 04 (DonderFormer) runs. Needs the run folder and its training split's normalization (`features/<id>/normalization/<split>.npz` plus the split manifest), not the rest of the feature cache.

`bpm` and `offset` only set the TJA's beat grid (bar lines, scroll speed, how cleanly notes snap); note timing follows the audio either way. Leave them as `"auto"` to estimate both from the audio (printed per song; watch for half/double tempo, and bar lines can land a beat off), or set the song's real values: `offset` is minus the time in seconds of a downbeat. Relative `audio` paths are read from the repo root. `level` is the Oni star level DonderFormer conditions on; the CNN-LSTM ignores it.

In [ ]:
import torch

from donderformer.paths import DataPaths, outputs_dir
from donderformer.song_generation import export_songs, load_saved_run

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
paths = DataPaths()

RUNS = [                                   # one chart per run and song
    outputs_dir() / "runs" / "donderformer" / "<run folder>",
    # outputs_dir() / "runs" / "taikonation" / "<run folder>",
]
SONGS = [
    dict(audio="data/music/my_song.ogg", title="My Song", bpm="auto", offset="auto", level=9),
]
ALIAS = "new_songs"                        # names the export folder
VERSION = "clean"                          # "clean" (back-to-back notes removed) or "raw"
SEED = 0

In [ ]:
runs = {}
for folder in RUNS:
    run = load_saved_run(folder, paths, device=device)
    if run.label in runs:
        raise ValueError(f"Two runs are both labelled {run.label}")
    runs[run.label] = run                  # e.g. donderformer_base_oni_v2; the .tja name and in-game title
    print(f"{run.label}: {run.run_dir}")

In [ ]:
export_dir = export_songs(SONGS, runs, ALIAS, version=VERSION, seed=SEED)